# Room Booking Assistant — technical walkthrough

Every section runs against the real modules in `app/`. Run it from the repository root with
`uv run jupyter nbconvert --execute --to notebook --inplace doc/walkthrough.ipynb`.
Section 6 talks to OpenAI and needs `OPENAI_API_KEY` in `.env`; the committed copy keeps its outputs.

| Technology | Role in the solution | Why this one |
|---|---|---|
| FastAPI | login, REST, chat API, static page | typed validation, OpenAPI at `/docs`, dependency injection for the current user |
| SQLAlchemy 2 · SQLite / Postgres | persistence | one model for local SQLite and Railway Postgres; the no-double-booking rule is a primary key |
| pwdlib (Argon2) · PyJWT | authentication | what the FastAPI documentation recommends today |
| LangChain `create_agent` on LangGraph | tool-calling agent | middleware for the confirmation gate, hidden runtime context for the user, checkpointed threads |
| OpenAI `gpt-6-luna`, Responses API | language understanding | the cheapest current model with function calling; switchable with `OPENAI_MODEL` |
| pytest | tests and evals | rules, API and agent wiring with a scripted model; evals with the real model |
| Docker · Railway · GitHub Actions | delivery | one image, managed Postgres, CI on every push

## 1. Setup: a throwaway database and a fixed clock

The clock is injected, so "now" is Tuesday 22 September 2026, 09:00 in Montevideo for every example.

In [1]:
import json
import os
import sys
import tempfile
from pathlib import Path

if Path.cwd().name == "doc":
    os.chdir(Path.cwd().parent)
sys.path.insert(0, str(Path.cwd()))

from app.booking import BookingError, BookingService
from app.config import Settings
from app.db import create_session_factory, seed
from tests.support import FIXED_NOW, FakeClock, booking_request

workspace = Path(tempfile.mkdtemp())
settings = Settings(jwt_secret="walkthrough-notebook-secret-not-for-production", database_url=f"sqlite:///{(workspace / 'rules.db').as_posix()}")
session_factory = create_session_factory(settings.sqlalchemy_database_url)
seed(session_factory, settings.seed_password)
service = BookingService(session_factory, settings, FakeClock(FIXED_NOW))
print("Office time:", FIXED_NOW.isoformat())
print("Rooms:", service.rooms())

Office time: 2026-09-22T09:00:00-03:00
Rooms: [{'room': 'A', 'capacity': 4}, {'room': 'B', 'capacity': 6}, {'room': 'C', 'capacity': 8}, {'room': 'D', 'capacity': 12}, {'room': 'E', 'capacity': 20}]


## 2. The rules live in `BookingService`, not in the prompt

Every violation is a `BookingError` with a code and the data the assistant needs to explain it.

In [2]:
print("Created:", service.create(1, booking_request()))

attempts = {
    "30 people in room B": booking_request(attendees=30),
    "three and a half hours": booking_request(start="2026-09-23T13:00", end="2026-09-23T16:30"),
    "starts at 10:15": booking_request(start="2026-09-23T10:15", end="2026-09-23T11:00"),
    "overlaps the booking above": booking_request(start="2026-09-23T11:00", end="2026-09-23T12:00"),
    "yesterday": booking_request(start="2026-09-21T10:00", end="2026-09-21T11:00"),
    "after closing": booking_request(start="2026-09-23T19:30", end="2026-09-23T20:30"),
    "blank title": booking_request(start="2026-09-23T15:00", end="2026-09-23T16:00", title="  "),
}
for label, request in attempts.items():
    try:
        service.create(2, request)
    except BookingError as error:
        print(f"{label:28} -> {json.dumps(error.as_dict())}")

Created: {'booking_id': 1, 'room': 'B', 'start': '2026-09-23T10:00', 'end': '2026-09-23T11:30', 'title': 'Interview with John Doe', 'attendees': 4}
30 people in room B          -> {"ok": false, "error": "CAPACITY_EXCEEDED", "message": "Room B holds at most 6 people.", "capacity": 6}
three and a half hours       -> {"ok": false, "error": "TOO_LONG", "message": "A booking lasts at most 3 hours.", "max_hours": 3}
starts at 10:15              -> {"ok": false, "error": "NOT_ALIGNED", "message": "Bookings start and end on the hour or the half hour."}
overlaps the booking above   -> {"ok": false, "error": "SLOT_TAKEN", "message": "Room B is already booked from 10:00 to 11:30.", "conflict": {"start": "2026-09-23T10:00", "end": "2026-09-23T11:30"}}
yesterday                    -> {"ok": false, "error": "IN_THE_PAST", "message": "The booking would start in the past.", "now": "2026-09-22T09:00"}
after closing                -> {"ok": false, "error": "OUTSIDE_BUSINESS_HOURS", "message": "Rooms can

## 3. No double booking, even under concurrency

Each 30-minute slot is a row with primary key `(room_id, slot_start)`. Below, the second writer's check is skipped to simulate a request whose check ran before the first one committed: the database still refuses it.

In [3]:
service.check_create = lambda request: None
try:
    service.create(2, booking_request())
except BookingError as error:
    print("Second writer:", error.code, "-", error.message)
finally:
    del service.check_create

Second writer: SLOT_TAKEN - Room B was just booked by someone else for that time.


## 4. Schedules and data minimization

User2 books room B too. User1 sees it only as occupied: the title written by User2 never reaches User1's LLM context.

In [4]:
service.create(2, booking_request(start="2026-09-23T14:00", end="2026-09-23T15:00", title="Secret merger talks"))
schedule = service.room_schedule(1, "B", service.parse_local_datetime("2026-09-23T00:00"), service.parse_local_datetime("2026-09-23T23:59"))
for entry in schedule["ranges"]:
    print(entry)

{'start': '2026-09-23T08:00', 'end': '2026-09-23T10:00', 'status': 'free'}
{'start': '2026-09-23T10:00', 'end': '2026-09-23T11:30', 'status': 'occupied', 'mine': True, 'booking_id': 1, 'title': 'Interview with John Doe'}
{'start': '2026-09-23T11:30', 'end': '2026-09-23T14:00', 'status': 'free'}
{'start': '2026-09-23T14:00', 'end': '2026-09-23T15:00', 'status': 'occupied', 'mine': False}
{'start': '2026-09-23T15:00', 'end': '2026-09-23T20:00', 'status': 'free'}


## 5. Tools: what the model sees, and what it does not

The user reaches the tools through `ToolRuntime` context. It is not an argument, so it is not in the schema the model receives and no prompt can change it.

In [5]:
from app.agent import build_tools

for agent_tool in build_tools(service):
    properties = agent_tool.tool_call_schema.model_json_schema().get("properties", {})
    print(f"{agent_tool.name:22} {list(properties)}")

list_available_rooms   ['start', 'end', 'attendees']
get_room_schedule      ['room', 'start', 'end']
list_my_bookings       []
create_booking         ['room', 'start', 'end', 'title', 'attendees']
cancel_booking         ['booking_id']


## 6. The agent end to end, with the confirmation gate

`create_agent` runs four middlewares: a dynamic system prompt, structured tool errors, `HumanInTheLoopMiddleware` (it pauses a write only when a dry run says it will succeed) and a limit of six model calls per message. This cell talks to the real model through the HTTP API.

In [6]:
import logging

from fastapi.testclient import TestClient

from app.main import create_app
from tests.support import login_headers

agent_settings = settings.model_copy(update={"database_url": f"sqlite:///{(workspace / 'agent.db').as_posix()}"})
app = create_app(agent_settings, clock=FakeClock(FIXED_NOW))
logging.getLogger("httpx2").setLevel(logging.WARNING)
client = TestClient(app)
headers = login_headers(client, "User1")


def say(message, conversation_id=None):
    reply = client.post("/chat/messages", json={"conversation_id": conversation_id, "message": message}, headers=headers).json()
    print("User1:     ", message)
    if reply["reply"]:
        print("Assistant: ", reply["reply"])
    for action in reply["pending_actions"]:
        print("Confirm?   ", action["summary"])
    return reply


first = say("Which rooms are free tomorrow from 10:00 to 11:00 for 6 people?")
second = say('Book room C then, title "Sprint planning"', first["conversation_id"])

2026-09-23 09:35:49,557 INFO app.agent thread=1:25639d62f99a4414bf4282fc9acee6e4 tool=list_available_rooms ok


User1:      Which rooms are free tomorrow from 10:00 to 11:00 for 6 people?
Assistant:  Tomorrow, September 23, from 10:00 to 11:00, rooms B, C, D, and E are free for 6 people. Room B is the smallest option.


User1:      Book room C then, title "Sprint planning"
Confirm?    Book room C · Wed 23 Sep, 10:00–11:00 · Sprint planning · 6 attendees


In [7]:
done = client.post("/chat/decisions", json={"conversation_id": second["conversation_id"], "approve": True}, headers=headers).json()
print("Assistant: ", done["reply"])
print("Bookings of User1:", client.get("/bookings/mine", headers=headers).json())

failure = say("Book room A tomorrow from 12:00 to 13:00 for 10 people, title Workshop")

2026-09-23 09:35:53,304 INFO app.agent thread=1:25639d62f99a4414bf4282fc9acee6e4 tool=create_booking ok


Assistant:  Room C is booked for 6 people tomorrow, September 23, from 10:00 to 11:00. Booking ID: 1.
Bookings of User1: [{'booking_id': 1, 'room': 'C', 'start': '2026-09-23T10:00', 'end': '2026-09-23T11:00', 'title': 'Sprint planning', 'attendees': 6}]


2026-09-23 09:36:01,063 INFO app.agent thread=1:8e60cb3810c84db993a878a3a0c3f18c tool=list_available_rooms ok


User1:      Book room A tomorrow from 12:00 to 13:00 for 10 people, title Workshop
Assistant:  La sala A tiene capacidad para 4 personas, así que no alcanza para 10. La sala D está disponible mañana, 23 de septiembre, de 12:00 a 13:00 y tiene capacidad para 12. ¿Quieres que reserve la sala D?


## 7. The HTTP API

The same service is reachable without the LLM; the full OpenAPI documentation is at `/docs`.

In [8]:
for path, methods in app.openapi()["paths"].items():
    print(f"{', '.join(method.upper() for method in methods):10} {path}")

GET        /health
POST       /auth/login
GET        /auth/me
GET        /rooms
GET        /rooms/{room_id}/schedule
GET        /bookings/mine
POST       /chat/messages
POST       /chat/decisions


## 8. Tests and evals

Unit tests script the model and run in seconds without a key. Evals use the real model and assert on the tools called and the final database state, not on wording.

In [9]:
import subprocess

unit = subprocess.run([sys.executable, "-m", "pytest", "-q"], capture_output=True, text=True)
print(unit.stdout.strip().splitlines()[-1])
evals = subprocess.run([sys.executable, "-m", "pytest", "-m", "eval", "-q"], capture_output=True, text=True)
print(evals.stdout.strip().splitlines()[-1])

85 passed, 11 deselected in 17.52s


11 passed, 85 deselected in 48.91s


## 9. Observability and deployment

- Logs: every refused tool call is logged with its error code; unexpected tool failures are logged with the traceback.
- Tracing: set `LANGSMITH_TRACING=true` and `LANGSMITH_API_KEY` to send every model call, tool call and interrupt to LangSmith without code changes.
- Deployment: one Docker image on Railway with managed Postgres; `/health` is the healthcheck; CI runs lint and unit tests on every push.